In [1]:
!nvidia-smi

Fri Jun 19 06:39:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q unsloth
!pip install -q transformers datasets trl peft accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 756.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [3]:
import unsloth
print("Unsloth installed successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully!


In [1]:
from unsloth import FastLanguageModel
print("Imports successful")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Imports successful


In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-4-mini-instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.6.8: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.31k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [3]:
print(type(model))

<class 'transformers.models.phi3.modeling_phi3.Phi3ForCausalLM'>


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [5]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(200064, 3072, padding_idx=200029)
        (layers): ModuleList(
          (0): Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_proj): Linear4bit(in

In [7]:
data = [
    {"question": "What is 2+2?", "answer": "4"},
    {"question": "Capital of India?", "answer": "New Delhi"},
    {"question": "5 + 7?", "answer": "12"},
    {"question": "Largest planet?", "answer": "Jupiter"},
    {"question": "Who wrote Hamlet?", "answer": "William Shakespeare"},
    {"question": "Water formula?", "answer": "H2O"},
    {"question": "Capital of France?", "answer": "Paris"},
    {"question": "10 - 3?", "answer": "7"},
    {"question": "Fastest land animal?", "answer": "Cheetah"},
    {"question": "Square of 6?", "answer": "36"},
]

In [8]:
from datasets import Dataset

def format_func(example):
    return {
        "text": f"### Instruction:\n{example['question']}\n\n### Response:\n{example['answer']}"
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_func)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [11]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=10,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
        report_to="none",

        # ✅ ADD THIS (fixes your error)
        save_strategy="no",
        save_steps=999999,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

In [12]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 4 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


Step,Training Loss
1,1.840434
2,1.967038
3,1.605906
4,1.285460
5,1.351192
6,0.825805
7,0.774798
8,0.862194
9,0.973238
10,0.858086


TrainOutput(global_step=10, training_loss=1.2344151556491851, metrics={'train_runtime': 8.7616, 'train_samples_per_second': 4.565, 'train_steps_per_second': 1.141, 'total_flos': 7985395654656.0, 'train_loss': 1.2344151556491851, 'epoch': 3.4})

In [13]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "What is 2+2?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=50) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is 2+2?4


In [14]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

Unsloth: Restored added_tokens_decoder metadata in lora_adapter/tokenizer_config.json.


('lora_adapter/tokenizer_config.json',
 'lora_adapter/chat_template.jinja',
 'lora_adapter/tokenizer.json')

In [15]:
!ls lora_adapter

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [16]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "What is 2+2?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=50) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is 2+2?4


In [17]:
ls


lora_adapter/  outputs/  sample_data/  unsloth_compiled_cache/


In [18]:
!ls lora_adapter

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [20]:
%%writefile README.md
# QLoRA Fine-tuning on Phi-4-mini (Unsloth)

## Project Overview
This project demonstrates a minimal QLoRA fine-tuning pipeline using Unsloth on Phi-4-mini.

## Steps Done
- Loaded Phi-4-mini in 4-bit precision
- Applied LoRA adapters (QLoRA)
- Created a 10-sample dummy dataset
- Trained for exactly 10 steps using SFTTrainer
- Verified training loss changes
- Tested inference after training
- Saved LoRA adapter weights

## Output
- Training completed successfully in 10 steps
- Loss decreased during training
- Model still produces valid outputs after fine-tuning

## Files
- phi4_qlora_training.ipynb
- lora_adapter/
- README.md

Writing README.md


In [21]:
!ls

lora_adapter  outputs  README.md  sample_data  unsloth_compiled_cache


In [22]:
!cat README.md

# QLoRA Fine-tuning on Phi-4-mini (Unsloth)

## Project Overview
This project demonstrates a minimal QLoRA fine-tuning pipeline using Unsloth on Phi-4-mini.

## Steps Done
- Loaded Phi-4-mini in 4-bit precision
- Applied LoRA adapters (QLoRA)
- Created a 10-sample dummy dataset
- Trained for exactly 10 steps using SFTTrainer
- Verified training loss changes
- Tested inference after training
- Saved LoRA adapter weights

## Output
- Training completed successfully in 10 steps
- Loss decreased during training
- Model still produces valid outputs after fine-tuning

## Files
- phi4_qlora_training.ipynb
- lora_adapter/
- README.md
